# Indoor67 — Classical baseline
Original method by Spencer Kolodny (`859926a`): 16x16 RGB thumbnail plus color histograms,
training-only scaling, and a linear SVM. Codex adapted the method to a reusable `.py`
module and shared split. This notebook calls that implementation, then explains results.
It uses labeled training/validation images only; committed outputs must remain empty.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/dataset.py").is_file() and (p / "configs/baseline.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from src.dataset import load_training_records
from src.classical import image_features, run_experiment
DATA_DIR = ROOT / "data/raw"
IMAGE_ROOT = DATA_DIR / "images"
train_df, val_df, class_names, split_metadata = load_training_records(DATA_DIR, ROOT / "splits")
print(f"Train: {len(train_df):,}; validation: {len(val_df):,}; classes: {len(class_names)}")

## 1. Understand the feature recipe
A 16x16 RGB thumbnail contributes 768 values. Three normalized 16-bin color histograms
contribute 48 more: 816 total. The SVM pipeline fits StandardScaler on fitting rows only.
The fixed recipe lives in `configs/classical.yaml`; no hyperparameter search occurs here.

In [ ]:
example = image_features(IMAGE_ROOT / train_df.iloc[0].file_path)
assert example.shape == (816,) and np.isfinite(example).all()
print("Feature dimensions:", len(example))
print("Each channel histogram sums to:", example[768:].reshape(3, 16).sum(axis=1))

## 2. Load a recorded run, or explicitly execute a new one
By default this notebook reads local artifacts, falling back to tracked compact results.
To fit a new run, first reserve a fresh ID and create its pre-run note. Then set
`AITP_RUN_CLASSICAL=1`, `AITP_CLASSICAL_RUN_ID=YOUR_NEW_ID`, and `AITP_RUN_OWNER=YOUR_NAME`
before launching Jupyter. Existing run IDs are never overwritten or re-registered.

In [ ]:
RUN_ID = os.environ.get("AITP_CLASSICAL_RUN_ID", "E010-classical-integration-v1")
OWNER = os.environ.get("AITP_RUN_OWNER", "unspecified (set AITP_RUN_OWNER)")
if os.environ.get("AITP_RUN_CLASSICAL", "0") == "1":
    run_experiment(ROOT / "configs/classical.yaml", RUN_ID, OWNER)
local = ROOT / "outputs" / RUN_ID
tracked = ROOT / "experiments/metrics" / RUN_ID
ARTIFACT_DIR = local if (local / "summary.json").is_file() else tracked
if not (ARTIFACT_DIR / "summary.json").is_file():
    raise FileNotFoundError("No recorded run. Follow README.md to create a new classical run.")
summary = json.loads((ARTIFACT_DIR / "summary.json").read_text())
assert summary["split_sha256"] == split_metadata["split_sha256"]
print("Reading:", ARTIFACT_DIR)
print("Code commit:", summary["code_commit"])
display(pd.read_csv(ARTIFACT_DIR / "metrics.csv"))

## 3. Classification report and confusion matrix
Rows are true classes; columns are predicted classes. These are validation metrics,
not Kaggle test scores. The dummy provides a trivial reference; the classical SVM
provides a comparison with learned ImageNet representations.

In [ ]:
display(pd.read_csv(ARTIFACT_DIR / "rgb_hist_linear_svm_classification_report.csv", index_col=0))
cm = pd.read_csv(ARTIFACT_DIR / "rgb_hist_linear_svm_confusion_matrix.csv", index_col=0).loc[class_names, class_names]
fig, ax = plt.subplots(figsize=(16, 14))
plot = ax.imshow(cm.to_numpy(), cmap="Blues")
ax.set(xticks=range(len(class_names)), yticks=range(len(class_names)),
       xticklabels=class_names, yticklabels=class_names,
       xlabel="Predicted class", ylabel="True class", title="Classical SVM validation confusion")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.setp(ax.get_yticklabels(), fontsize=7)
fig.colorbar(plot, ax=ax)
plt.tight_layout()
plt.show()

## 4. Inspect a reproducible sample of validation errors
This is an explanation/diagnostic view of labeled validation examples. It must not be
used to inspect test images and assign test labels. Eight errors are a sample, not a
causal explanation for all failures.

In [ ]:
predictions = pd.read_csv(ARTIFACT_DIR / "val_predictions.csv", dtype=str)
assert predictions.id.is_unique and set(predictions.id) == set(val_df.id)
truth = val_df.set_index("id").label
assert predictions.label.tolist() == predictions.id.map(truth).tolist()
errors = predictions.loc[predictions.label != predictions.rgb_hist_linear_svm_label]
sample = errors.sample(min(8, len(errors)), random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax in axes.flat:
    ax.axis("off")
for ax, row in zip(axes.flat, sample.itertuples()):
    with Image.open(IMAGE_ROOT / row.file_path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"True: {row.label}\nPredicted: {row.rgb_hist_linear_svm_label}", fontsize=9)
plt.tight_layout()
plt.show()

## 5. Reproducibility and interpretation
The runner saves the training-only scaler/classifier, feature settings, class order,
split hashes, source commit, environment, and validation predictions. It checks that
reloading each locally created checkpoint reproduces predictions. Read the run note
before interpreting small differences across operating systems or package versions.

Spencer's original results are preserved separately in `experiments/metrics/spencer-original/`.
The integration rerun has its own IDs and provenance. Do not replace historical metrics.
Clear all notebook outputs before committing; generated image galleries stay local.

In [ ]:
print("Checkpoint reload predictions match:", summary["checkpoint_reload_predictions_match"])
print("Original method commit:", summary["source_method_commit"])
print("Execution owner:", summary["execution_owner"])
print("Warnings:", summary["warnings"])